# 머신 러닝 교과서 - 파이토치편

## 좋은 훈련 데이터셋 만들기 – 데이터 전처리


### 목차

1. 누락된 데이터 다루기

    1\) 테이블 형태 데이터에서 누락된 값 식별
    2\) 누락된 값이 있는 샘플이나 특성 제외
    3\) 누락된 값 대체

2. 범주형 데이터 다루기

    1\) 순서가 있는 특성 매핑
    2\) 클래스 레이블 인코딩
    3\) 순서가 없는 특성에 원-핫 인코딩 적용

3. 데이터셋을 훈련 데이터셋과 테스트 데이터셋으로 나누기

4. 특성 스케일 맞추기

In [ ]:
from IPython.display import Image

### 1. 누락된 데이터 다루기

* ```NaN``` (not a number): 숫자 타입인데, 유효한 값이 아님
    - 0/0, 누락된 수치형 데이터
* ```NULL```: 데이터베이스 용어로 값이 없음을 의미
* ```None```: python에서 값이 없음을 의미하는 내장 객체(object)


#### 1) 테이블 형태 데이터에서 누락된 값 식별


In [ ]:
import pandas as pd
from io import StringIO
import sys

csv_data = \
'''A,B,C,D
1.0,2.0,3.0,4.0
5.0,6.0,,8.0
10.0,11.0,12.0,'''

df = pd.read_csv(StringIO(csv_data))
df

In [ ]:
df.isnull().sum()

In [ ]:
# `values` 속성으로 넘파이 배열을 얻을 수 있습니다
df.values

#### 2) 누락된 값이 있는 샘플이나 특성 제외


In [ ]:
# 누락된 값이 있는 행을 삭제합니다
df.dropna(axis=0)

In [ ]:
# 누락된 값이 있는 열을 삭제합니다
df.dropna(axis=1)

In [ ]:
import numpy as np
df.loc[3] = np.array([np.nan]*4)
df

In [ ]:
# 모든 열이 NaN인 행을 삭제합니다
df.dropna(axis=0, how='all')
df

In [ ]:
# NaN 아닌 값이 네 개보다 작은 행을 삭제합니다
df.dropna(thresh=4)

In [ ]:
# 특정 열에 NaN이 있는 행만 삭제합니다(여기서는 'C'열)
df.dropna(subset=['C'])

#### 3) 누락된 값 대체
  

In [ ]:
df.fillna(df.mean())

In [ ]:
df.bfill() # 뒤의 값으로 채우기 (backfill)

In [ ]:
df.ffill() # 앞의 값으로 채우기 (forward fill)

In [ ]:
df.ffill(axis=1)

---

### 2. 범주형 데이터 다루기


#### 판다스를 사용한 범주형 데이터 인코딩
지금까지는 **수치형 데이터**만 사용했습니다. 실제 데이터셋은 하나 이상의 범주형 특성이 포함된 경우가 많습니다.

**범주형 데이터**를 어떻게 **수치형 데이터**로 관리할 수 있는지에 대해 살펴봅니다.


In [ ]:
import pandas as pd

df = pd.DataFrame([['green', 'M', 10.1, 'class2'],
                   ['red', 'L', 13.5, 'class1'],
                   ['blue', 'XL', 15.3, 'class2']])

df.columns = ['color', 'size', 'price', 'classlabel']
df

#### 1) 순서가 있는 특성 매핑


In [ ]:
size_mapping = {'XL': 3,
                'L': 2,
                'M': 1}

df['size'] = df['size'].map(size_mapping)
df

In [ ]:
size_mapping.items()

In [ ]:
inv_size_mapping = {v: k for k, v in size_mapping.items()}
df['size'].map(inv_size_mapping)

#### 2) 클래스 레이블 인코딩

* 많은 머신 러닝 라이브러리는 클래스 레이블이 정수로 인코딩되었을 것이라고 가정

In [ ]:
import numpy as np

# 클래스 레이블을 문자열에서 정수로 바꾸기 위해
# 매핑 딕셔너리를 만듭니다
class_mapping = {label: idx for idx, label in enumerate(np.unique(df['classlabel']))}
class_mapping

In [ ]:
# 클래스 레이블을 문자열에서 정수로 바꿉니다
df['classlabel'] = df['classlabel'].map(class_mapping)
df

In [ ]:
# 클래스 레이블을 거꾸로 매핑합니다
inv_class_mapping = {v: k for k, v in class_mapping.items()}
df['classlabel'] = df['classlabel'].map(inv_class_mapping)
df

#### 3) 순서가 없는 특성에 원-핫 (one-hot) 인코딩 적용
* ```color``` 특성을 ```blue```, ```green```, ```red``` 특성으로 변환
* 이진 값을 사용하여 특정 샘플의 color를 나타냄
* Ex) 특정 sample이 ```blue```라면 ```blue=1```, ```green=0```, ```red=0```로 표현 

In [ ]:
df

In [ ]:
# color를 정수로 매핑하면?
color_mapping = {'blue': 0, 'green': 1, 'red': 2}
df_color = df[['color', 'size', 'price']].copy()
df_color['color'] = df_color['color'].map(color_mapping)
df_color

* 이 경우 red > green > blue 라는 의미를 갖게 됨
* 이를 해결하기 위해 원-핫 인코딩 (one-hot encoding) 사용

In [ ]:
# 판다스를 사용한 원-핫 인코딩
# 문자열 열만 변환
pd.get_dummies(df[['price', 'color', 'size']])

In [ ]:
# 특정 열에 대해서 원-핫 인코딩 적용
pd.get_dummies(df[['price', 'color', 'size']], columns=['size'])

* 다중 공선성 (multicollinearity)
    - 독립변수들 사이에 강한 상관관계가 존재하는 현상
    - 특정 알고리즘들에서 문제가 발생할 수 있음 (역행렬 만들기가 어려워짐)
    - Ex. ```blue = !(green or red)```

In [ ]:
# get_dummies에서 다중 공선성 문제 처리
pd.get_dummies(df[['price', 'color', 'size']], drop_first=True)

### 3. 데이터셋을 훈련 데이터셋과 테스트 데이터셋으로 나누기

* 훈련 데이터셋(train set): 모델을 학습시키는 데 사용
* 테스트 데이터셋(test set): 학습에 사용하지 않은 데이터로, 모델의 성능을 평가하는 데 사용
* 데이터가 특정 순서로 정렬되어 있을 수 있으므로 **무작위로 섞은 후** 나눔

#### 붓꽃(iris) 데이터셋 살펴보기

* 붓꽃 150송이의 꽃받침(sepal)과 꽃잎(petal)의 길이/너비를 측정한 데이터
* 측정값 4개(특성)로 붓꽃의 품종 3가지(타깃)를 분류하는 데 사용

| 열 이름 | 의미 |
|---|---|
| sepal_length | 꽃받침 길이 (cm) |
| sepal_width | 꽃받침 너비 (cm) |
| petal_length | 꽃잎 길이 (cm) |
| petal_width | 꽃잎 너비 (cm) |
| species | 품종 (setosa, versicolor, virginica) → **타깃** |

In [ ]:
import seaborn as sns

df_iris = sns.load_dataset('iris')
df_iris.head()

In [ ]:
print('shape :', df_iris.shape)
print()
print(df_iris['species'].value_counts())   # 품종별 개수

In [ ]:
df_iris.describe()

In [ ]:
# 데이터가 품종 순서대로 정렬되어 있음: 0~49행 setosa, 50~99행 versicolor, 100~149행 virginica
df_iris.iloc[[0, 49, 50, 99, 100, 149]]

* 앞에서부터 순서대로 70%를 자르면 훈련 데이터에는 virginica가 거의 없고, 테스트 데이터는 virginica만 남음

In [ ]:
X, y = df_iris.drop(columns='species'), df_iris['species']

# 잘못된 예: 섞지 않고 앞 70%를 훈련 데이터로 사용
print(y[:105].value_counts())    # 훈련 데이터
print()
print(y[105:].value_counts())    # 테스트 데이터

In [ ]:
# 올바른 예: 무작위로 뽑아서 나누기
X_test = X.sample(frac=0.3, random_state=0)   # 30%를 무작위로 뽑아 테스트 데이터로 사용
X_train = X.drop(X_test.index)                # 나머지 70%는 훈련 데이터로 사용
y_test, y_train = y[X_test.index], y[X_train.index]

print(X_train.shape, X_test.shape)

In [ ]:
# 훈련/테스트 데이터 모두 세 품종이 골고루 섞여 있음
print(y_train.value_counts())
print()
print(y_test.value_counts())

* 무작위로 나누면 품종이 섞이긴 하지만, 품종별 비율이 정확히 같지는 않음 (훈련 데이터: virginica 39, versicolor 32)
* **층화 분할 (stratified split)**: label(품종)마다 따로 7:3으로 나눈 뒤 합치면, 훈련/테스트 데이터의 label 비율이 원본과 같아짐
    - 데이터가 적거나 label 비율이 불균형할 때 특히 중요

In [ ]:
# 층화 분할: 품종마다 30%씩 뽑아 테스트 데이터로 사용
test_index = []
for label in y.unique():                       # ['setosa', 'versicolor', 'virginica']
    X_label = X[y == label]                    # 해당 품종의 데이터만 선택
    X_label_test = X_label.sample(frac=0.3, random_state=0)
    test_index += list(X_label_test.index)     # 뽑힌 행의 인덱스를 모아둠

X_test = X.loc[test_index]                     # 모아둔 인덱스의 행: 테스트 데이터
X_train = X.drop(test_index)                   # 나머지 행: 훈련 데이터
y_test, y_train = y[X_test.index], y[X_train.index]

print(X_train.shape, X_test.shape)

In [ ]:
# 훈련/테스트 데이터 모두 세 품종이 정확히 같은 비율 (35:35:35, 15:15:15)
print(y_train.value_counts())
print()
print(y_test.value_counts())

### 4. 특성 스케일 맞추기

* 대부분의 머신 러닝과 최적화 알고리즘은 특성 간의 스케일이 같을 때 성능이 좋음
* 정규화(normalization): 특성의 스케일을 $[0,1]$에 맞추는 것
* 표준화(standardization): 데이터가 표준정규분포와 같은 특징을 갖도록 함
    - 정규화보다 이상치에 덜 민감
* ⚠️ 최솟값/최댓값, 평균/표준편차는 **훈련 데이터로만** 계산하고, 테스트 데이터에도 같은 값을 사용 (테스트 데이터는 학습 때 볼 수 없는 데이터이기 때문)


#### 1) 최소-최대 스케일 변환(min-max scaling): 정규화
$$
x^{(i)}_{norm} = \frac{x^{(i)}-x_{min}}{x_{max}-x_{min}}
$$

In [ ]:
# 훈련 데이터의 최솟값, 최댓값 계산 (열마다)
x_min = X_train.min()
x_max = X_train.max()

X_train_norm = (X_train - x_min) / (x_max - x_min)
X_test_norm = (X_test - x_min) / (x_max - x_min)   # 테스트 데이터도 훈련 데이터의 값으로 변환
X_train_norm.head()

#### 2) 표준화

$$
x^{(i)}_{std}=\frac{x^{(i)}-\mu_x}{\sigma_x}
$$

In [ ]:
# 훈련 데이터의 평균, 표준편차 계산 (열마다)
mean = X_train.mean()
std = X_train.std()

X_train_std = (X_train - mean) / std
X_test_std = (X_test - mean) / std                 # 테스트 데이터도 훈련 데이터의 값으로 변환
X_train_std.head()

A visual example:

In [ ]:
ex = np.array([0, 1, 2, 3, 4, 5])

print('standardized:', (ex - ex.mean()) / ex.std())

# 판다스는 기본적으로 ddof=1를 사용합니다(샘플 표준 편차).
# 반면 넘파이 std 메서드는 ddof=0를 사용합니다(모집단 표준 편차).

# 정규화합니다
print('normalized:', (ex - ex.min()) / (ex.max() - ex.min()))